# Lesson 7.3: Color Histogram and Enhancement
## Biomedical Image Processing - Color Image Processing

### Topics:
- Color image histograms
- Per-channel histogram equalization
- HSI-based intensity equalization
- Color image enhancement techniques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a 200x200 low-contrast color image
# Pixel values are concentrated in a narrow range (80-170)
np.random.seed(42)
img = np.zeros((200, 200, 3), dtype=np.uint8)

# Red-ish region (top-left quadrant)
img[0:100, 0:100, 0] = np.random.randint(120, 170, (100, 100))   # R high
img[0:100, 0:100, 1] = np.random.randint(80, 120, (100, 100))    # G low
img[0:100, 0:100, 2] = np.random.randint(80, 110, (100, 100))    # B low

# Green-ish region (top-right quadrant)
img[0:100, 100:200, 0] = np.random.randint(80, 120, (100, 100))  # R low
img[0:100, 100:200, 1] = np.random.randint(120, 170, (100, 100)) # G high
img[0:100, 100:200, 2] = np.random.randint(80, 110, (100, 100))  # B low

# Blue-ish region (bottom-left quadrant)
img[100:200, 0:100, 0] = np.random.randint(80, 110, (100, 100))  # R low
img[100:200, 0:100, 1] = np.random.randint(80, 110, (100, 100))  # G low
img[100:200, 0:100, 2] = np.random.randint(120, 170, (100, 100)) # B high

# Yellow-ish region (bottom-right quadrant)
img[100:200, 100:200, 0] = np.random.randint(120, 160, (100, 100))  # R high
img[100:200, 100:200, 1] = np.random.randint(120, 160, (100, 100))  # G high
img[100:200, 100:200, 2] = np.random.randint(80, 110, (100, 100))   # B low

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title("Low-Contrast Color Test Image")
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Image shape: {img.shape}, dtype: {img.dtype}")
print(f"Value range: [{img.min()}, {img.max()}]")
print("Notice: values are compressed into a narrow range (80-170) -> low contrast")

## 1. Color Image Histograms

A color image has **three histograms** — one for each channel (R, G, B).

Understanding the distribution of each channel helps us:
- Detect low contrast (values clustered in a narrow range)
- Identify color casts (one channel dominates)
- Decide which enhancement technique to apply

In [ ]:
# Plot the image and its per-channel histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: the image
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[0].axis('off')

# Right: overlapping histograms for R, G, B
colors = ['red', 'green', 'blue']
labels = ['Red', 'Green', 'Blue']
for i, (c, lbl) in enumerate(zip(colors, labels)):
    axes[1].hist(img[:, :, i].ravel(), bins=256, range=(0, 256),
                 color=c, alpha=0.5, label=lbl)
axes[1].set_title("Per-Channel Histograms")
axes[1].set_xlabel("Pixel Value")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].set_xlim([0, 256])

plt.tight_layout()
plt.show()

print("All three channels are concentrated between ~80 and ~170.")
print("This confirms the image has low contrast.")

## 2. Per-Channel Histogram Equalization

Apply histogram equalization **independently** to each channel (R, G, B).

The equalization transform maps pixel values so that the output histogram is approximately uniform:

$$s_k = (L - 1) \sum_{j=0}^{k} p(r_j), \quad k = 0, 1, \ldots, L-1$$

where $p(r_j)$ is the normalized histogram and $L = 256$.

**Pros:** Simple to implement, improves contrast in each channel.

**Cons:** Can cause **color shifts** because channels are processed independently — the relative balance between R, G, B may change.

In [ ]:
def histogram_equalize(channel):
    """
    Apply histogram equalization to a single grayscale channel (uint8).
    Returns the equalized channel (uint8).
    """
    hist, _ = np.histogram(channel.ravel(), bins=256, range=(0, 256))
    cdf = hist.cumsum()
    # Normalize CDF to [0, 255]
    cdf_normalized = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min())
    cdf_normalized = cdf_normalized.astype(np.uint8)
    return cdf_normalized[channel]

# Per-channel histogram equalization
eq_r = histogram_equalize(img[:, :, 0])
eq_g = histogram_equalize(img[:, :, 1])
eq_b = histogram_equalize(img[:, :, 2])
img_eq_rgb = np.stack([eq_r, eq_g, eq_b], axis=2)

# Display original vs equalized
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img)
axes[0, 0].set_title("Original")
axes[0, 0].axis('off')

axes[0, 1].imshow(img_eq_rgb)
axes[0, 1].set_title("Per-Channel Equalized")
axes[0, 1].axis('off')

# Histograms before
colors = ['red', 'green', 'blue']
for i, c in enumerate(colors):
    axes[1, 0].hist(img[:, :, i].ravel(), bins=256, range=(0, 256),
                    color=c, alpha=0.5)
axes[1, 0].set_title("Original Histograms")
axes[1, 0].set_xlim([0, 256])

# Histograms after
for i, c in enumerate(colors):
    axes[1, 1].hist(img_eq_rgb[:, :, i].ravel(), bins=256, range=(0, 256),
                    color=c, alpha=0.5)
axes[1, 1].set_title("Equalized Histograms")
axes[1, 1].set_xlim([0, 256])

plt.suptitle("Per-Channel Histogram Equalization", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Per-channel equalization spreads values across [0, 255] for each channel.")
print("Notice: the color balance may shift because channels are equalized independently.")

## 3. HSI-Based Intensity Equalization

A better approach: convert to **HSI**, equalize only the **Intensity** channel, and convert back.

This preserves the original **hue** and **saturation** while improving contrast.

Steps:
1. Convert RGB → HSI
2. Equalize only the I channel
3. Scale RGB proportionally to match the new intensity
4. Convert back to uint8

In [ ]:
# HSI-based intensity equalization
img_float = img.astype(np.float64) / 255.0

# Compute intensity: I = (R + G + B) / 3
intensity = np.mean(img_float, axis=2)

# Equalize the intensity channel
# Convert intensity to uint8 for histogram equalization
intensity_u8 = (intensity * 255).astype(np.uint8)
intensity_eq = histogram_equalize(intensity_u8)
new_intensity = intensity_eq.astype(np.float64) / 255.0

# Scale RGB channels proportionally: new_RGB = old_RGB * (new_I / old_I)
scale = np.where(intensity > 0.001, new_intensity / intensity, 1.0)
img_eq_hsi = np.clip(img_float * scale[:, :, np.newaxis], 0, 1)
img_eq_hsi = (img_eq_hsi * 255).astype(np.uint8)

# Compare all three: original, per-channel equalized, HSI-based equalized
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img)
axes[0].set_title("Original")

axes[1].imshow(img_eq_rgb)
axes[1].set_title("Per-Channel Equalized\n(color shift possible)")

axes[2].imshow(img_eq_hsi)
axes[2].set_title("HSI Intensity Equalized\n(preserves hue)")

for ax in axes:
    ax.axis('off')

plt.suptitle("Comparison: Per-Channel vs HSI-Based Equalization", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("HSI-based equalization improves contrast while better preserving the original colors.")

## 4. Color Image Enhancement Techniques

### 4.1 Gamma Correction

Gamma correction applies a nonlinear transformation to adjust brightness:

$$g(x,y) = c \cdot f(x,y)^{\gamma}$$

- $\gamma < 1$: brightens dark regions (expands low values)
- $\gamma > 1$: darkens bright regions (compresses high values)

### 4.2 Saturation Enhancement

Increasing saturation makes colors more vivid. In HSI space, we multiply the saturation channel by a factor.

In [ ]:
# Gamma correction on color images
img_norm = img.astype(np.float64) / 255.0

gamma_low = np.clip(img_norm ** 0.5, 0, 1)     # gamma < 1: brighten
gamma_high = np.clip(img_norm ** 2.0, 0, 1)    # gamma > 1: darken

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow((gamma_low * 255).astype(np.uint8))
axes[0].set_title("γ = 0.5 (Brighten)")

axes[1].imshow(img)
axes[1].set_title("Original (γ = 1.0)")

axes[2].imshow((gamma_high * 255).astype(np.uint8))
axes[2].set_title("γ = 2.0 (Darken)")

for ax in axes:
    ax.axis('off')

plt.suptitle("Gamma Correction on Color Image", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Saturation enhancement
# Convert to float and compute HSI components
img_f = img.astype(np.float64) / 255.0
R, G, B = img_f[:, :, 0], img_f[:, :, 1], img_f[:, :, 2]

intensity = (R + G + B) / 3.0

# Increase saturation by scaling distance from gray axis
sat_factor = 1.8  # > 1 = more vivid colors

# For each pixel, move it away from the gray point (I, I, I)
enhanced = np.zeros_like(img_f)
for ch in range(3):
    enhanced[:, :, ch] = np.clip(
        intensity + sat_factor * (img_f[:, :, ch] - intensity), 0, 1
    )

enhanced_u8 = (enhanced * 255).astype(np.uint8)

# Desaturate for comparison
desat_factor = 0.3
desaturated = np.zeros_like(img_f)
for ch in range(3):
    desaturated[:, :, ch] = np.clip(
        intensity + desat_factor * (img_f[:, :, ch] - intensity), 0, 1
    )
desaturated_u8 = (desaturated * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(desaturated_u8)
axes[0].set_title(f"Desaturated (factor={desat_factor})")

axes[1].imshow(img)
axes[1].set_title("Original")

axes[2].imshow(enhanced_u8)
axes[2].set_title(f"Saturated (factor={sat_factor})")

for ax in axes:
    ax.axis('off')

plt.suptitle("Saturation Enhancement", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Saturation factor > 1 makes colors more vivid.")
print("Saturation factor < 1 moves colors toward gray (desaturation).")

## Summary

What we learned:
1. **Color histograms** = three separate histograms (R, G, B) reveal contrast and color distribution
2. **Per-channel histogram equalization** = equalizes each channel independently; may cause color shifts
3. **HSI-based intensity equalization** = equalize only intensity to preserve hue and saturation
4. **Gamma correction** = nonlinear brightness adjustment using $f^{\gamma}$
5. **Saturation enhancement** = scale the distance from the gray axis to make colors more/less vivid